# Object Tracking Project — Football Player Tracking
## Part 3: Tracking with the ByteTrack Algorithm

In the previous notebook (`03-BotSort.ipynb`) we implemented the full tracking-by-detection pipeline with the **BoT-SORT** algorithm (which also uses appearance/Re-ID features). In this notebook we repeat the same pipeline with **ByteTrack**.

### Key Difference from BoT-SORT

Unlike BoT-SORT, ByteTrack does not use a Re-ID network or appearance features — it relies purely on motion information. As a result:

```text
No heavy feature extraction
        ↓
Computational Cost ↓
FPS ↑
```

This makes ByteTrack a very attractive option for **real-time** systems; as we saw in the accuracy/speed comparisons when the algorithms were first introduced, among the fast, Re-ID-free methods, ByteTrack also offers better accuracy than something like plain SORT.

This notebook's structure mirrors the BoT-SORT notebook exactly (same SSD model as detector, same test video), with the difference that:
* We no longer need to set a `device` for the tracker or provide Re-ID weights (since ByteTrack has no Re-ID model).
* Creating the tracker is much simpler.


### Suppressing Warnings


In [1]:
import warnings
warnings.filterwarnings("ignore")

### Imports

This time we import `ByteTrack` (instead of `BotSort`) from the `boxmot` library.


In [2]:
import cv2
import torch
import numpy as np
from boxmot import ByteTrack

### Loading the Detector (the Same Trained SSD Model)

We read the SSD model saved in the first notebook (`best_model.pth`) and load it onto the appropriate `device`.


In [3]:
SSD_MODEL_PATH = "best_model.pth"

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.load(SSD_MODEL_PATH,weights_only=False)

### Setting the Model to Inference Mode

We move the model to `device` and set it to `eval()` mode so we're only doing inference.


In [5]:
model.to(device)
model.eval()

SSD(
  (backbone): SSDFeatureExtractorVGG(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (15): ReLU(inplace=

### Opening the Test Video File

We open the same sample video we used in the BoT-SORT notebook for tracking.


In [6]:
VIDEO_PATH = "football_players_detection/video/video2.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

### No Need for a Separate Device Setting

Unlike BoT-SORT, here we don't need to specify a separate `device_str` for the tracker, since ByteTrack has no Re-ID/feature-extractor network that would need a GPU. All of ByteTrack's computation (distance, matching, track management) is lightweight and CPU-based.


In [7]:
# device_str = "0" if torch.cuda.is_available() else "cpu"

### Creating the Tracker Instance

Unlike `BotSort`, which took several parameters (Re-ID weight path, device, half, with_reid), creating `ByteTrack` is much simpler and works fine with default settings.


In [8]:
tracker = ByteTrack()

## Running Tracking on the Video (Simple Version)

This loop follows exactly the same pattern as the previous notebook:

```text
Frame → Detector (SSD) → Detections [x1, y1, x2, y2, conf, class] → tracker.update() → Tracks + IDs
```

The key point here is that **the `tracker.update(detections, frame)` call is identical across all BoxMOT trackers** (whether ByteTrack or BoT-SORT) — we just need to swap out the tracker object, and the rest of the pipeline stays unchanged. This means we can easily try different tracking algorithms on the same detector output and compare them.


In [9]:
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # (H, W, Ch) >>> (batch,Ch, H, W)
    img_tensor = torch.from_numpy(frame).permute(2,0,1).unsqueeze(0).float().to(device) / 255.0

    with torch.no_grad():
        preds = model(img_tensor)[0]

    detections = []
    if "boxes" in preds:
        boxes  = preds["boxes"].cpu().numpy()
        scores = preds["scores"].cpu().numpy()
        labels = preds["labels"].cpu().numpy()
        for (x1, y1, x2, y2), score, label in zip(boxes, scores, labels):
            detections.append([x1, y1, x2, y2, float(score), int(label)])
    detections = np.array(detections)

    tracks = tracker.update(detections, frame)

    if tracks.size > 0:
        tracker.plot_results(frame, show_trajectories=False)

    cv2.imshow("tracking", frame)
    if cv2.waitKey(1) == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## Managing FPS and Displaying the Video at Its Real Speed

Just like in the previous notebook, we import `time` to measure per-frame processing time and keep the video's display speed in sync with its real FPS.


In [10]:
import time

### Improved Version with Proper Delay Calculation

Exactly the same logic as the BoT-SORT notebook: we read the video's real frame rate, measure processing time per frame and display the instantaneous processing FPS on the image, and compute the delay needed to keep the display speed in sync with the video's real FPS (clamped to a minimum of 1 ms so it's never negative).

Important note: since ByteTrack is lighter than BoT-SORT (no Re-ID), the processing FPS here will usually be **far higher** than the video's real FPS (25); without this delay management, the video would play much faster than real-time.

---

## Summary: BoT-SORT vs. ByteTrack

| Feature | BoT-SORT | ByteTrack |
|---|---|---|
| Motion | ✅ | ✅ |
| Appearance / Re-ID | ✅ | ❌ |
| Association accuracy | High | High |
| Computational cost | Higher | Lower |
| FPS | Lower | Higher |
| Re-identification (after occlusion) | ✅ Stronger | More limited |
| Suited to real-time | Depends on hardware | ⭐ Very well suited |

If **speed and real-time performance** are the priority, **ByteTrack** is a good fit. If **re-identification accuracy and reducing ID switches** matter more (e.g. when a player is temporarily occluded by another player), **BoT-SORT** is the better choice.

In the next notebook (`05-yolo_tracking.ipynb`) we replace SSD with **YOLO** as the detector, and see how the **Ultralytics** library combines fine-tuning the detector and running tracking (with the same two algorithms, BoT-SORT/ByteTrack) into a much simpler pipeline.


In [12]:
VIDEO_PATH = "football_players_detection/video/video2.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

target_fps = cap.get(cv2.CAP_PROP_FPS)
target_duration_ms = int((1 / target_fps) * 1000)

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # Record the start time for frame processing
    start_time = time.time()
    
    # (H, W, Ch) >>> (batch,Ch, H, W)
    img_tensor = torch.from_numpy(frame).permute(2,0,1).unsqueeze(0).float().to(device) / 255.0

    with torch.no_grad():
        preds = model(img_tensor)[0]

    detections = []
    if "boxes" in preds:
        boxes  = preds["boxes"].cpu().numpy()
        scores = preds["scores"].cpu().numpy()
        labels = preds["labels"].cpu().numpy()
        for (x1, y1, x2, y2), score, label in zip(boxes, scores, labels):
            detections.append([x1, y1, x2, y2, float(score), int(label)])
    detections = np.array(detections)

    tracks = tracker.update(detections, frame)

    if tracks.size > 0:
        tracker.plot_results(frame, show_trajectories=False)

    # Record the end time of processing
    end_time = time.time()
    
    # Calculate and display the FPS
    duration = end_time - start_time
    fps = 1 / duration
    cv2.putText(frame, f"FPS: {int(fps)}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    print(fps)

    
    # Calculate the required delay
    processing_time_ms = (end_time - start_time) * 1000
    wait_time = target_duration_ms - processing_time_ms
    if wait_time < 1:
        wait_time = 1
        
    cv2.imshow("tracking", frame)
    if cv2.waitKey(int(wait_time)) == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

49.88408796279778
33.4775675050085
36.696215156870636
43.952090035523796
46.85169174402109
44.024981368937034
56.439534414317436
55.58019718011237
60.04300336411137
55.175142729353574
56.411448246180335
53.69396402739551
51.5682547488781
51.4209493919184
53.048136999468795
57.51767642138175
54.50970810698412
44.099505835348545
46.9303257135824
50.4480821736568
54.758786359601025
57.92837511221601
54.236222101533606
58.44660897677076
43.323768502163965
52.27589301293716
46.14043540917241
51.77194346725915
60.84609693470471
46.627209462614225
46.37203285829584
42.31968519826455
47.98261128207475
55.0925235117953
58.44253706387248
59.45909471087736
60.524740616747714
49.18504620291758
53.823501482156374
55.23690621995707
56.994795559239584
57.764030243351556
58.151649174372984
57.78074114891859
59.3951031621281
49.75685679035779
51.98562256761112
57.20467533176034
53.1173334346466
53.85252616036464
53.844230201419826
59.16303213247948
57.37604990287544
54.978424433084285
54.38813247231516